# Source Profiling Framework

## Purpose

Automatically profiles every dataset in the OneLake landing zone.

## Outputs

- Table Statistics
- Column Statistics
- Null Analysis
- Duplicate Analysis
- Cardinality
- Data Analysis
- Relationship Checks

## Target

Lakehouse Profiling Folder

In [18]:
from pathlib import Path
import pandas as pd
import pyspark.sql.functions as F
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *
from datetime import datetime, timezone
import json
import uuid
import time

StatementMeta(, 79dcee32-a722-44cc-ae4e-11250c6e86ce, 24, Finished, Available, Finished, False)

In [2]:
LAKEHOUSE_ROOT = "/lakehouse/default/"
SOURCE_FOLDER = f"{LAKEHOUSE_ROOT}/Files/landing/olist/source_csv"
PROFILE_FOLDER = f"{LAKEHOUSE_ROOT}/Files/profiling"
REPORT_FOLDER = f"{PROFILE_FOLDER}/reports"
STATISTICS_FOLDER = f"{PROFILE_FOLDER}/statistics"
LOG_FOLDER = f"{PROFILE_FOLDER}/logs"

StatementMeta(, 79dcee32-a722-44cc-ae4e-11250c6e86ce, 4, Finished, Available, Finished, False)

# Profiling Run Metadata

In [ ]:
PROFILE_RUN_ID = str(uuid.uuid4())
PROFILE_START_TIME = datetime.now(timezone.utc)
PROFILE_START_PERF = time.perf_counter()

print("Profiling run ID: ", PROFILE_RUN_ID)
print("Profiling started at: ", PROFILE_START_TIME)

In [3]:
import os

for folder in [PROFILE_FOLDER, REPORT_FOLDER, STATISTICS_FOLDER, LOG_FOLDER]:
    os.makedirs(folder, exist_ok=True)

StatementMeta(, 79dcee32-a722-44cc-ae4e-11250c6e86ce, 5, Finished, Available, Finished, False)

In [4]:
csv_files = list(Path(SOURCE_FOLDER).glob("*.csv"))

print(len(csv_files))
print(csv_files)

StatementMeta(, 79dcee32-a722-44cc-ae4e-11250c6e86ce, 6, Finished, Available, Finished, False)

9
[PosixPath('/lakehouse/default/Files/landing/olist/source_csv/olist_customers_dataset.csv'), PosixPath('/lakehouse/default/Files/landing/olist/source_csv/olist_geolocation_dataset.csv'), PosixPath('/lakehouse/default/Files/landing/olist/source_csv/olist_order_items_dataset.csv'), PosixPath('/lakehouse/default/Files/landing/olist/source_csv/olist_order_payments_dataset.csv'), PosixPath('/lakehouse/default/Files/landing/olist/source_csv/olist_order_reviews_dataset.csv'), PosixPath('/lakehouse/default/Files/landing/olist/source_csv/olist_orders_dataset.csv'), PosixPath('/lakehouse/default/Files/landing/olist/source_csv/olist_products_dataset.csv'), PosixPath('/lakehouse/default/Files/landing/olist/source_csv/olist_sellers_dataset.csv'), PosixPath('/lakehouse/default/Files/landing/olist/source_csv/product_category_name_translation.csv')]


In [5]:
from pathlib import Path

datasets = {}

for file in csv_files:
    dataset_name = file.stem

    spark_path = (
        f"Files/landing/olist/source_csv/{file.name}"
    )

    datasets[dataset_name] = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .option("multiLine", True)
        .option("quote", '"')
        .option("escape", '"')
        .csv(spark_path)
    )

    print(f"Loaded: {dataset_name}")

StatementMeta(, 79dcee32-a722-44cc-ae4e-11250c6e86ce, 7, Finished, Available, Finished, False)

Loaded: olist_customers_dataset
Loaded: olist_geolocation_dataset
Loaded: olist_order_items_dataset
Loaded: olist_order_payments_dataset
Loaded: olist_order_reviews_dataset
Loaded: olist_orders_dataset
Loaded: olist_products_dataset
Loaded: olist_sellers_dataset
Loaded: product_category_name_translation


# Verifying if Datasets loaded correctly

In [6]:
for dataset_name,df in datasets.items():
    print("="*20)
    print(f"Dataset: {dataset_name}")
    print(f"Rows: {df.count():,}")
    print(f"Columns: {len(df.columns)}")

StatementMeta(, 79dcee32-a722-44cc-ae4e-11250c6e86ce, 8, Finished, Available, Finished, False)

Dataset: olist_customers_dataset
Rows: 99,441
Columns: 5
Dataset: olist_geolocation_dataset
Rows: 1,000,163
Columns: 5
Dataset: olist_order_items_dataset
Rows: 112,650
Columns: 7
Dataset: olist_order_payments_dataset
Rows: 103,886
Columns: 5
Dataset: olist_order_reviews_dataset
Rows: 99,224
Columns: 7
Dataset: olist_orders_dataset
Rows: 99,441
Columns: 8
Dataset: olist_products_dataset
Rows: 32,951
Columns: 9
Dataset: olist_sellers_dataset
Rows: 3,095
Columns: 4
Dataset: product_category_name_translation
Rows: 71
Columns: 2


# Generating table level profiling statistical summary

In [7]:
table_summary = []

for dataset_name, df in datasets.items():
    row_count = df.count()
    column_count = len(df.columns)

    table_summary.append({
        "table_name": dataset_name,
        "row_count": row_count,
        "column_count": column_count,
        "profiled_at": datetime.now()
    })

table_summary_df = pd.DataFrame(table_summary)

display(table_summary_df)

StatementMeta(, 79dcee32-a722-44cc-ae4e-11250c6e86ce, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d5a3c473-cde4-42fc-b2ce-45f857b0c499)

In [8]:
spark.createDataFrame(table_summary_df).write.mode("overwrite").format("delta").saveAsTable('profile_table_summary')

StatementMeta(, 79dcee32-a722-44cc-ae4e-11250c6e86ce, 10, Finished, Available, Finished, False)

# Column Level Profiling

In [9]:
import builtins

column_summary = []

for dataset_name, df in datasets.items():

    total_rows = df.count()

    for field in df.schema.fields:

        column_name = str(field.name)
        data_type = str(field.dataType)
        nullable = field.nullable

        escaped_name = column_name.replace("`", "``")
        column_ref = F.col(f"`{escaped_name}`")

        null_count = (
            df.filter(column_ref.isNull())
            .count()
        )

        distinct_count = (
            df.select(column_ref)
            .distinct()
            .count()
        )

        null_percentage = (
            builtins.round(
                (null_count / total_rows) * 100,
                2
            )
            if total_rows > 0
            else 0.0
        )

        column_summary.append({
            "table_name": dataset_name,
            "column_name": column_name,
            "data_type": data_type,
            "nullable": nullable,
            "row_count": total_rows,
            "null_count": null_count,
            "null_percentage": null_percentage,
            "distinct_values": distinct_count
        })

print("Column summary records created:", len(column_summary))

StatementMeta(, 79dcee32-a722-44cc-ae4e-11250c6e86ce, 11, Finished, Available, Finished, False)

Column summary records created: 52


In [10]:
column_summary_df = pd.DataFrame(column_summary)
display(column_summary_df.head(20))

StatementMeta(, 79dcee32-a722-44cc-ae4e-11250c6e86ce, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e3bb0c8f-a012-40ae-bf43-573d20b5b0dd)

In [11]:
spark.createDataFrame(column_summary_df).write.mode("overwrite").format("delta").saveAsTable("profile_column_summary")

StatementMeta(, 79dcee32-a722-44cc-ae4e-11250c6e86ce, 13, Finished, Available, Finished, False)

# Null Analysis

In [14]:
null_summary = []

for dataset_name, df in datasets.items():

    total_rows = df.count()

    for field in df.schema.fields:

        column_name = str(field.name)

        # Escape column names safely
        escaped_name = column_name.replace("`", "``")
        column_ref = F.col(f"`{escaped_name}`")

        null_count = (
            df.filter(column_ref.isNull())
            .count()
        )

        null_percentage = (
            builtins.round(
                (null_count / total_rows) * 100,
                2
            )
            if total_rows > 0
            else 0.0
        )

        not_null_count = total_rows - null_count

        not_null_percentage = builtins.round(
            100.0 - null_percentage,
            2
        )

        null_summary.append({
            "table_name": dataset_name,
            "column_name": column_name,
            "row_count": total_rows,
            "null_count": null_count,
            "null_percentage": null_percentage,
            "not_null_count": not_null_count,
            "not_null_percentage": not_null_percentage
        })

StatementMeta(, 79dcee32-a722-44cc-ae4e-11250c6e86ce, 17, Finished, Available, Finished, False)

In [16]:
null_summary_df = pd.DataFrame(null_summary)
display(
    null_summary_df.sort_values(
        "null_percentage",
        ascending=False
    )
)

StatementMeta(, 79dcee32-a722-44cc-ae4e-11250c6e86ce, 22, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 2a958b8b-444f-4141-b1da-589379687904)

In [17]:
spark.createDataFrame(null_summary_df)\
    .write\
    .mode("overwrite")\
    .format("delta")\
    .saveAsTable("profile_null_summary")

StatementMeta(, 79dcee32-a722-44cc-ae4e-11250c6e86ce, 23, Finished, Available, Finished, False)

# Duplicate and Key Analysis

In [20]:
BUSINESS_KEYS = {
    "olist_customers_dataset": ["customer_id"],
    "olist_geolocation_dataset": [
        "geolocation_zip_code_prefix",
        "geolocation_lat",
        "geolocation_lng",
        "geolocation_city",
        "geolocation_state"
    ],
    "olist_order_items_dataset": ["order_id", "order_item_id"],
    "olist_order_payments_dataset": [
        "order_id",
        "payment_sequential"
    ],
    "olist_order_reviews_dataset": ["review_id"],
    "olist_orders_dataset": ["order_id"],
    "olist_products_dataset": ["product_id"],
    "olist_sellers_dataset": ["seller_id"],
    "product_category_name_translation": [
        "product_category_name"
    ]
}

StatementMeta(, 79dcee32-a722-44cc-ae4e-11250c6e86ce, 26, Finished, Available, Finished, False)

In [22]:
duplicate_summary = []

for dataset_name, df in datasets.items():

    total_rows = df.count()

    # Exact duplicate rows
    distinct_rows = df.distinct().count()
    exact_duplicate_rows = total_rows - distinct_rows

    key_columns = BUSINESS_KEYS.get(dataset_name, [])

    missing_key_columns = [
        column_name
        for column_name in key_columns
        if column_name not in df.columns
    ]

    if missing_key_columns:
        raise ValueError(
            f"{dataset_name} is missing key columns: "
            f"{missing_key_columns}"
        )

    if key_columns:
        duplicate_key_groups = (
            df.groupBy(*key_columns)
              .count()
              .filter(F.col("count") > 1)
        )

        duplicate_key_group_count = (
            duplicate_key_groups.count()
        )

        duplicate_key_row_count = (
            duplicate_key_groups
            .select(
                F.sum(
                    F.col("count") - F.lit(1)
                ).alias("duplicate_rows")
            )
            .collect()[0]["duplicate_rows"]
        )

        duplicate_key_row_count = (
            int(duplicate_key_row_count)
            if duplicate_key_row_count is not None
            else 0
        )

        distinct_key_count = (
            df.select(*key_columns)
              .distinct()
              .count()
        )

    else:
        duplicate_key_group_count = None
        duplicate_key_row_count = None
        distinct_key_count = None

    exact_duplicate_percentage = (
        builtins.round(
            (exact_duplicate_rows / total_rows) * 100,
            4
        )
        if total_rows > 0
        else 0.0
    )

    duplicate_summary.append({
        "profile_run_id": PROFILE_RUN_ID,
        "table_name": dataset_name,
        "business_key": ", ".join(key_columns),
        "row_count": total_rows,
        "distinct_row_count": distinct_rows,
        "exact_duplicate_row_count": exact_duplicate_rows,
        "exact_duplicate_percentage": exact_duplicate_percentage,
        "distinct_business_key_count": distinct_key_count,
        "duplicate_key_group_count": duplicate_key_group_count,
        "duplicate_business_key_row_count": duplicate_key_row_count
    })

StatementMeta(, 79dcee32-a722-44cc-ae4e-11250c6e86ce, 28, Finished, Available, Finished, False)

In [23]:
duplicate_summary_spark_df = spark.createDataFrame(duplicate_summary)

display(
    duplicate_summary_spark_df
    .orderBy(
        F.col("exact_duplicate_row_count").desc(),
        F.col("duplicate_business_key_row_count").desc_nulls_last()
    )
)

(
    duplicate_summary_spark_df
    .write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("profile_duplicate_summary")
)

StatementMeta(, 79dcee32-a722-44cc-ae4e-11250c6e86ce, 29, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 147224c9-0837-4d37-abf4-9d9b8ce6691c)

In [24]:
def show_duplicate_keys(dataset_name: str, limit: int = 20):
    df = datasets[dataset_name]
    key_columns = BUSINESS_KEYS[dataset_name]

    duplicate_keys = (
        df.groupBy(*key_columns)
        .count()
        .filter(F.col("count") > 1)
        .orderBy(F.col("count").desc())
    )

    display(duplicate_keys.limit(limit))

StatementMeta(, 79dcee32-a722-44cc-ae4e-11250c6e86ce, 30, Finished, Available, Finished, False)

In [26]:
show_duplicate_keys("olist_geolocation_dataset")

StatementMeta(, 79dcee32-a722-44cc-ae4e-11250c6e86ce, 32, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7ed6d210-89d0-4f1a-9d9a-09e0789d60d8)

# Business-key Completeness

Measure null values in expected source business-key columns.

In [27]:
key_quality_summary = []

for dataset_name, key_columns in BUSINESS_KEYS.items():

    df = datasets[dataset_name]
    total_rows = df.count()

    null_condition = None

    for column_name in key_columns:

        column_ref = F.col(column_name)

        condition = (
            column_ref.isNull()
            | (F.trim(column_ref) == "")
        )

        null_condition = (
            condition
            if null_condition is None
            else null_condition | condition
        )

    rows_with_missing_key = (
        df.filter(null_condition)
        .count()
    )

    missing_key_percentage = (
        builtins.round(
            (rows_with_missing_key / total_rows) * 100,
            4
        )
        if total_rows > 0
        else 0.0
    )

    key_quality_summary.append({
        "profile_run_id": PROFILE_RUN_ID,
        "table_name": dataset_name,
        "business_key": ", ".join(key_columns),
        "row_count": total_rows,
        "rows_with_missing_key": rows_with_missing_key,
        "missing_key_percentage": missing_key_percentage
    })

StatementMeta(, 79dcee32-a722-44cc-ae4e-11250c6e86ce, 33, Finished, Available, Finished, False)

In [28]:
key_quality_spark_df = spark.createDataFrame(key_quality_summary)

display(
    key_quality_spark_df
    .orderBy(
        F.col("rows_with_missing_key").desc()
    )
)

(
    key_quality_spark_df
    .write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("profile_key_quality_summary")
)

StatementMeta(, 79dcee32-a722-44cc-ae4e-11250c6e86ce, 34, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3feedc21-383a-4064-a9b2-0c1691825237)

In [32]:
PROFILE_END_TIME = datetime.now(timezone.utc)

PROFILE_DURATION_SECONDS = builtins.round(
    time.perf_counter() - PROFILE_START_PERF,
    2
)

TOTAL_TABLES = len(datasets)

TOTAL_COLUMNS = builtins.sum(
    len(df.columns)
    for df in datasets.values()
)

StatementMeta(, 79dcee32-a722-44cc-ae4e-11250c6e86ce, 38, Finished, Available, Finished, False)

In [33]:
profiling_run_record = [{
    "profile_run_id": PROFILE_RUN_ID,
    "started_at_utc": PROFILE_START_TIME,
    "completed_at_utc": PROFILE_END_TIME,
    "duration_seconds": PROFILE_DURATION_SECONDS,
    "dataset_count": len(csv_files),
    "table_count": TOTAL_TABLES,
    "column_count": TOTAL_COLUMNS,
    "status": "SUCCESS"
}]

profiling_run_df = spark.createDataFrame(profiling_run_record)

display(profiling_run_df)

StatementMeta(, 79dcee32-a722-44cc-ae4e-11250c6e86ce, 39, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 231757f5-9497-4a31-bc3e-6c99993d3d44)

In [34]:
(
    profiling_run_df
    .write
    .mode("append")
    .format("delta")
    .saveAsTable("profile_run_history")
)

StatementMeta(, 79dcee32-a722-44cc-ae4e-11250c6e86ce, 40, Finished, Available, Finished, False)